# LeJEPA Pretraining
This script trains the following lejepa-trained models on the ImageNet-100 dataset:
- ResNet-50
- ViT-S/16

## 1. Imports

### 1.1 Install packages
**Note:** Remember to turn on internet in kaggle

In [ ]:
!pip install timm --quiet
!pip install -q git+https://github.com/galilai-group/lejepa.git

### 1.2 Imports libraries

In [ ]:
import os
import time
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.amp import GradScaler
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torchvision.ops import MLP
import timm
import shutil
import matplotlib.pyplot as plt
import lejepa
from PIL import Image as PILImage

print(lejepa.__file__)
print(timm.__version__)
print(torch.cuda.is_bf16_supported())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print("ok")

## 2. Globals

In [ ]:
SEED = 42

# dataset
IMAGENET_ROOT_TRAIN = "/kaggle/input/datasets/gpukaggle1/imagenet-100/imagenet100/imagenet1k/train"
IMAGENET_ROOT_VAL = "/kaggle/input/datasets/gpukaggle1/imagenet-100/imagenet100_val/imagenet1k/val_filtered"

NUM_CLASSES = 100
NUM_WORKERS = min(4, os.cpu_count() or 2)
PREFETCH_FACTOR = 2 # preload batches in advance

# architecture/model
ARCH = "resnet50" # resnet50 or vit_small_patch16_224

# training
EPOCHS = 40 # real objective is 100
# batch size and gradient accumulation
# restnet50: batch 256
# vit-s/16: batch 128 + accum 2 = 256 (vit is more expensive)
BATCH_SIZE = 256 if ARCH == "resnet50" else 128
ACCUM_STEPS = 1 if ARCH == "resnet50" else 2
EFFECTIVE_BS = BATCH_SIZE * ACCUM_STEPS # 256

# optmization
# from https://github.com/galilai-group/lejepa README
LR_BASE = 5e-4
WEIGHT_DECAY = 5e-2 if "vit" in ARCH else 5e-4
WARMUP_EPOCHS = max(4, EPOCHS // 10) # 10% 
ETA_MIN = LR_BASE / 1000
BETAS = (0.9, 0.999)

# mixed precision
USE_AMP = True
AMP_DTYPE = "float16" # bfloat16 

# lejepa sigreg
# from lejepa paper: (page 13)
# We thus recommend to use 17 integration points, 
# an integration domain of [−5,5], 
# and 1024 slices as starting points.
LAMBDA = 0.05
NUM_SLICES = 1024
NUM_POINTS = 17   # projector dim Epps-Pulley
PROJ_DIM = 128 # from table 1d of the paper
T_MAX = 5.0 # range -5, 5

# ResNet: only 2 global views, N_LOCAL = 0
# ViT: 2 global (224px) + 6 local (96px), dynamic_img_size=True
# from Algorithm 2 
IS_VIT = "vit" in ARCH
N_GLOBAL = 2
N_LOCAL = 6 if IS_VIT else 0
N_TOTAL = N_GLOBAL + N_LOCAL

# checkpointing
CKPT_DIR = f"/kaggle/working/checkpoints/lejepa_{ARCH}"
CKPT_EVERY = 5 # every 5 epochs

# layer hooks for PCA and SAS
HOOK_LAYERS = {
    "resnet50": ["layer1", "layer2", "layer3", "layer4"],
    "vit_small_patch16_224": ["blocks.2", "blocks.5", "blocks.8", "blocks.11"]
}

# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()

# cudnn 
cudnn.deterministic = False
cudnn.benchmark = True
  
# checkpoint folder
os.makedirs(CKPT_DIR, exist_ok=True)
if os.path.exists("/kaggle/input/datasets/gpucolabkagg/resnet-checkpoints/checkpoints-resnet"):
    shutil.copytree(
        "/kaggle/input/datasets/gpucolabkagg/resnet-checkpoints/checkpoints-resnet",
        CKPT_DIR,
        dirs_exist_ok=True  
    )
print(f"Architecture: {ARCH}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE} x {ACCUM_STEPS} = {EFFECTIVE_BS}")
print(f"LR: {LR_BASE} -> {ETA_MIN:.2e} (cosine)")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Warmup epochs: {WARMUP_EPOCHS}")
print(f"Lambda: {LAMBDA}")
print(f'Views: {N_GLOBAL} global + {N_LOCAL} local = {N_TOTAL} total')
print(f'SIGReg slices: {NUM_SLICES}')
print(f"AMP dtype: {AMP_DTYPE}")
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"Layers for hooks: {HOOK_LAYERS[ARCH]}")
print(f"Device: {DEVICE}")
print(f"GPUs available: {N_GPUS}")

if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM (GB):", props.total_memory / 1024**3)
    print("Compute capability:", props.major, ".", props.minor, sep="")

print(os.listdir(CKPT_DIR))

## 3. Utils

### 3.1 Seed for reproducibility

In [ ]:
# reproducibility
def set_seed(seed: int = 42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed(SEED)
print(f"Seed: {SEED}")

### 3.2 Checkpoints

In [ ]:
def save_checkpoint(epoch, backbone, projector, optimizer, scheduler, scaler, metrics):
    """
    Save checkpoint
    1. checkpoint_epXXX.pt - checkpoint with all the state dicts (backbone + projector)
    2. backbone_epXXX.pt - backbone only
    3. history.json - training history
    """
    state = {
        "epoch": epoch,
        "arch": ARCH,
        "seed": SEED,
        "lambda": LAMBDA, "n_global": N_GLOBAL, "n_local": N_LOCAL,
        "backbone_state": backbone.state_dict(),
        "projector_state": projector.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else {},
        "metrics": metrics,
        "hook_layers": HOOK_LAYERS[ARCH],
        "embed_dim": EMBED_DIM,
    }
    path = os.path.join(CKPT_DIR, f"checkpoint_ep{epoch:03d}.pt")
    torch.save(state, path)

    # backbone
    backbone_path = os.path.join(CKPT_DIR, f"backbone_ep{epoch:03d}.pt")
    torch.save({
        'backbone_state': backbone.state_dict(),'arch': ARCH, 
        'embed_dim': EMBED_DIM, 'epoch': epoch, 
        'seed': SEED, 'hook_layers': HOOK_LAYERS[ARCH],
        'val_acc1': None,  # for linear_probe (after)
    }, backbone_path)

    print(f"Checkpoint ep{epoch:03d} saved to {path}")

def load_checkpoint(backbone, projector, optimizer, scheduler, scaler, ckpt_path):
    """
    Load checkpoint and resume epoch
    """
    state = torch.load(ckpt_path, map_location=DEVICE)
    backbone.load_state_dict(state["backbone_state"])
    projector.load_state_dict(state["projector_state"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    if scaler is not None and state.get("scaler"):
        scaler.load_state_dict(state["scaler"])
    print(f"Resumed from epoch {state['epoch']}")
    return state["epoch"]

def find_latest_checkpoint(ckpt_dir):
    """
    Find the checkpoint_epXXX.pt with XXX recent
    """
    if not os.path.isdir(ckpt_dir):
        return None
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("checkpoint_ep") and f.endswith(".pt")]
    if not files:
        return None
    files.sort(key=lambda f: int(f.replace("checkpoint_ep", "").replace(".pt", "")))
    return os.path.join(ckpt_dir, files[-1])

print("ok")

## 4. Data

### 4.1 Upload the data for network training

In [ ]:
# transform

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# photometric transformations
_photo = [
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.RandomApply([
        T.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))], p=0.5),
    T.RandomSolarize(threshold=128, p=0.2),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
]

# global views 
global_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.3, 1.0)),
    *_photo
])

# local views
local_transform = T.Compose([
    T.RandomResizedCrop(96, scale=(0.05, 0.3)),
    *_photo
])

# validation
val_transform = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("ok")

In [ ]:
# multicrop

class MultiCropTransform:
    """
    N_GLOBALS global views + N_LOCAL local views of the same image
    """
    def __init__(self, n_global, n_local):
        self.n_global = n_global
        self.n_local = n_local 

    def __call__(self, img):
        views = [global_transform(img) for _ in range(self.n_global)]
        views += [local_transform(img) for _ in range(self.n_local)]
        return views

def multicrop_collate(batch):
    """
    Collate for the multi-crop dataset
    input: list of (views_list, label)
    output: (list of views tensors, labels), tensor shape (B, C, H, W)
    """
    views_list, labels = zip(*batch)
    n_views = len(views_list[0])
    batched = [
        torch.stack([views_list[i][v] for i in range(len(views_list))])
        for v in range(n_views)
    ]
    return batched, torch.tensor(labels)
    

multicrop = MultiCropTransform(N_GLOBAL, N_LOCAL)

print("ok")

In [ ]:
# dataset and dataloader
train_dataset = ImageFolder(IMAGENET_ROOT_TRAIN, transform=multicrop)
val_dataset = ImageFolder(IMAGENET_ROOT_VAL, transform=val_transform)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True, # avoid incomplete batch with AdamW
    persistent_workers=True if NUM_WORKERS > 0 else False,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    collate_fn=multicrop_collate
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)

print(f"Train: {len(train_dataset)} images, {len(train_loader)} batch")
print(f"Val: {len(val_dataset)} images, {len(val_loader)} batch")
print(f"Classes: {len(train_dataset.classes)}")
print(f'Mapping example: {dict(list(train_dataset.class_to_idx.items())[:3])}')

## 5. SIGReg and LeJEPA loss

**SIGReg**: lejepa.univariate.EppsPulley -> lejepa.multivariate.SlicingUnivariateTest

The loss works on the projected embedding (from the projector), instead of the raw embedding of the backbone

### 5.1 SigREG
Using lejepa official github repository

In [ ]:
# epps-pulley univariate test 
univariate_test = lejepa.univariate.EppsPulley(n_points=NUM_POINTS, t_max=T_MAX)

# slicing univariate test 
# SIGReg
# input (*, N, D) output: scalar
sigreg_fn = lejepa.multivariate.SlicingUnivariateTest(
    univariate_test=univariate_test,
    num_slices=NUM_SLICES
)

print("ok")

### 5.2 LeJEPA loss
From Algorithm 2 of the paper

In [ ]:
def LeJEPA_loss(all_embs, lamb):
    """
    Loss LeJEPA = (1 - lamb) * inv_loss + lamb * sigreg_loss 

    Algorithm 2:
    # embedding of global views
    g_emb = forward(torch . cat(glob_views) )
    # embedding of local views
    # if resnet: skip with a_emb=g_emb
    a_emb = forward(torch . cat( all_views ) )
    # LeJEPA loss
    centers = g_emb.view(−1, bs, K) .mean(0)
    a_emb = a_emb.view(−1, bs, K)
    sim = (centers − a_emb) .square() .mean()
    sigreg = mean(SIGReg(emb, global_step) for emb in a_emb)
    return (1−lambd)∗sim + lambd∗sigreg

    inv_loss: centers = only GLOBAL views mean and then MSE
    sigreg_loss: sigreg = mean_v[SIGReg(view_v)]
    """
    # stack embedding from all views
    proj_stack = torch.stack(all_embs, dim=0) # (V, B, PROJ_DIM)

    # inv loss
    centers = proj_stack[:N_GLOBAL].mean(dim=0) # (B, PROJ_DIM)
    inv_loss = (centers - proj_stack).pow(2).mean() # MSE
    
    # sigreg
    sigreg_loss = sigreg_fn(proj_stack) # scalar per batch

    # final loss
    lejepa_loss = (1 - lamb) * inv_loss + lamb * sigreg_loss 

    return lejepa_loss, inv_loss, sigreg_loss

print("ok")

## 6. Backbone

### 6.1 Build the backbone

In [ ]:
def build_backbone(arch):
    """
    Backbone without classification head (num_classes=0)
    """
    if arch == "resnet50":
        backbone = timm.create_model("resnet50", pretrained=False, num_classes=0)
        embed_dim = backbone.num_features # 2048
    elif arch == "vit_small_patch16_224":
        backbone = timm.create_model(
            "vit_small_patch16_224",
            pretrained=False,
            num_classes=0,
            global_pool="", # (B, N+1, D) all tokens
            dynamic_img_size=True # for local view 96x96
        )
        embed_dim = backbone.num_features # 384
    else:
        raise ValueError(f"Unknown architecture: {arch}")

    return backbone, embed_dim

def get_cls_embedding(backbone, x, arch):
    """Extract CLS/Global Average Pooling embedding from backbone"""
    out = backbone(x)
    if "vit" in arch and out.ndim == 3:
        return out[:, 0, :] # CLS token (B, D)
    return out # ResNet (B, D)

class Encoder(nn.Module):
    """
    Backbone + projector with a forward(x) pass
    """
    def __init__(self, arch):
        super().__init__()
        self.backbone, self.embed_dim = build_backbone(arch)
        #self.proj = MLP(self.embed_dim, [2048, 2048, PROJ_DIM], norm_layer=nn.BatchNorm1d)
        self.arch = arch

    def forward(self, x):
        emb = get_cls_embedding(self.backbone, x, self.arch) # (N, embed_dim)
        #return self.proj(emb) # (N, PROJ_DIM)
        return emb


set_seed(SEED)
_encoder = Encoder(ARCH).to(DEVICE)
backbone = _encoder.backbone
EMBED_DIM = _encoder.embed_dim

# projector (separate module)
projector = MLP(EMBED_DIM, [2048, 2048, PROJ_DIM], norm_layer=nn.BatchNorm1d).to(DEVICE)

if N_GPUS > 1:
    model = nn.DataParallel(_encoder)
    print(f"DataParallel on {N_GPUS} GPUs")
else:
    model = _encoder

# parameters
n_params = sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in projector.parameters())
n_train  = (sum(p.numel() for p in model.parameters() if p.requires_grad) + sum(p.numel() for p in projector.parameters() if p.requires_grad))
print(f"Architecture: {ARCH}")
print(f"Embedding dim: {EMBED_DIM}")
print(f"Number of parameters: {n_params/1e6:.1f}M")
print(f"Number of trainable parameters: {n_train/1e6:.1f}M")

### 6.2 Optimizer and scheduler

In [ ]:
def build_optimizer_and_scheduler(model, projector):
    """
    AdamW + linear warmup + cosine annealing
    lr 5e-4, lr_min = lr/1000, weight decay model specific
    weight decay to non-bias and non-norm parameters only
    """
    decay, no_decay = [], []
    for module in (model, projector):
        for name, param in module.named_parameters():
            if not param.requires_grad:
                continue
            if param.ndim <= 1 or name.endswith(".bias"):
                no_decay.append(param)  # bias, LayerNorm, BatchNorm
            else:
                decay.append(param)

    param_groups = [
        {"params": decay, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay, "weight_decay": 0.0}
    ]

    optimizer = optim.AdamW(param_groups, lr=LR_BASE, betas=BETAS)

    # linear warmup
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1e-3, end_factor=1.0, total_iters=WARMUP_EPOCHS
    )
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS-WARMUP_EPOCHS, eta_min=ETA_MIN
    )
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[WARMUP_EPOCHS]
    )
    return optimizer, scheduler

optimizer, scheduler = build_optimizer_and_scheduler(model, projector)
scaler = GradScaler("cuda", enabled=USE_AMP) if AMP_DTYPE == "float16" else None
print(f"Optimizer: {optimizer}")
print(f"Scheduler: {scheduler}")
print(f"Scaler: {scaler}")

## 7. Train

### 7.1 Resume training from checkpoints

In [ ]:
_resume_ckpt = find_latest_checkpoint(CKPT_DIR)

if _resume_ckpt is not None:
    START_EPOCH = load_checkpoint(backbone, projector, optimizer, scheduler, scaler, _resume_ckpt)
    _history_path = os.path.join(CKPT_DIR, "history.json")
    if os.path.exists(_history_path):
        with open(_history_path) as f:
            history = json.load(f)
        # if history recorded epochs next to START EPOCH
        history = [r for r in history if r["epoch"] <= START_EPOCH]
        print(f"Restore History: {len(history)} epochs already registered")
    else:
        history = []
        print("No history.json found")
else:
    START_EPOCH = 0
    history = []
    print("No checkpoint found: training from scratch")

print(f"Start epoch: {START_EPOCH}")

### 7.2 Training functions

In [ ]:
def train_one_epoch(model, projector, loader, optimizer, scaler):
    """
    Runs one LeJEPA training epoch

    For each batch of N_TOTAL views (order: [global, ..., local, ...]):
        views are grouped bu resolution (global 224px, local 96px)
        projection are re-concatenated in the original view order and split back per-view
        LeJEPA loss: inv loss + SIGReg 
        backward + gradient clipping + step every ACCUM_STEPS batches
    """
    model.train()
    projector.train()
    
    total_lejepa_loss, total_inv_loss, total_sigreg_loss = 0.0, 0.0, 0.0
    optimizer.zero_grad()

    amp_dtype = torch.bfloat16 if AMP_DTYPE == "bfloat16" else torch.float16
    
    num_batches = len(loader)
    last_accum_steps = num_batches % ACCUM_STEPS  # if not divisible, last step is smaller
    last_batch_idx_to_accum = num_batches - last_accum_steps
    accum_steps = ACCUM_STEPS
    
    all_params = list(model.parameters()) + list(projector.parameters())
    
    pbar = tqdm(loader, total=num_batches, desc="Training", leave=False)
    for step, (views, labels) in enumerate(pbar):
        # views: list of N_TOTAL tensors (B, C, H, W)
        # labels NOT used: LeJEPA is SSL
        views = [v.to(DEVICE, non_blocking=True) for v in views]
        B = views[0].size(0)

        last_batch = (step == num_batches - 1)
        if last_accum_steps > 0 and step >= last_batch_idx_to_accum:
            accum_steps = last_accum_steps  # renormalize for last step
        need_update = last_batch or (step + 1) % accum_steps == 0
        
        with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=USE_AMP):
            global_cat = torch.cat(views[:N_GLOBAL], dim=0) #(N_GLOBAL*B, C, 224, 224)
            global_emb = model(global_cat)

            if N_LOCAL > 0:
                local_cat = torch.cat(views[N_GLOBAL:], dim=0) # (N_LOCAL*B, C, 96, 96)
                local_emb = model(local_cat)
                raw_cat = torch.cat([global_emb, local_emb], dim=0)
            else:
                raw_cat = global_emb

            all_proj = projector(raw_cat) # V views
            all_embs = list(all_proj.split(B, dim=0)) # list V x (B, PROJ_DIM)

            lejepa_loss, inv_loss, sigreg_loss = LeJEPA_loss(all_embs, LAMBDA)
            lejepa_loss = lejepa_loss / accum_steps

        if scaler is not None:
            scaler.scale(lejepa_loss).backward()
        else:
            lejepa_loss.backward()

        if need_update:
            if scaler is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
                optimizer.step()
            optimizer.zero_grad()

        with torch.no_grad():
            total_lejepa_loss += lejepa_loss.item() * accum_steps
            total_inv_loss += inv_loss.item()
            total_sigreg_loss += sigreg_loss.item()
        
        pbar.set_postfix(loss=f"{total_lejepa_loss/(step+1):.3f}")
    
    return {
        "lejepa_loss": total_lejepa_loss / num_batches,
        "inv_loss": total_inv_loss / num_batches,
        "sigreg_loss": total_sigreg_loss / num_batches
    }

print("ok")

### 7.3 Training loop

In [ ]:
print(f"Start training from epoch {START_EPOCH}/{EPOCHS} - {ARCH}")
print(f"Views: {N_GLOBAL} global + {N_LOCAL} local = {N_TOTAL}")
print(f"Effective batch size: {EFFECTIVE_BS}")

for epoch in range(START_EPOCH, EPOCHS):
    t0 = time.time()

    # train
    train_metrics = train_one_epoch(model, projector, train_loader, optimizer, scaler)

    # scheduler step
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    elapsed = time.time() - t0

    # log
    record = {
        "epoch": epoch+1,
        "lr": current_lr,
        "lejepa_loss": train_metrics["lejepa_loss"],
        "inv_loss": train_metrics["inv_loss"],
        "sigreg_loss": train_metrics["sigreg_loss"],
        "elapsed_s": elapsed
    }
    history.append(record)

    print(
        f"Ep {epoch+1:3d}/{EPOCHS} "
        f"| lr {current_lr:.2e} "
        f"| lejepa loss {train_metrics['lejepa_loss']:.4f} "
        f"(inv loss {train_metrics['inv_loss']:.4f} "
        f" sigreg loss {train_metrics['sigreg_loss']:.4f}) "
        f"| {elapsed:.0f}s"
    )

    # checkpoint
    if (epoch + 1) % CKPT_EVERY == 0 or (epoch + 1) == EPOCHS:
        save_checkpoint(epoch+1, backbone, projector, optimizer, scheduler, scaler, record)
        history_path = os.path.join(CKPT_DIR, "history.json")
        with open(history_path, "w") as f:
            json.dump(history, f, indent=4)


# save complete history
history_path = os.path.join(CKPT_DIR, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=4)

print("Training complete!")
print(f"Final LeJEPA loss: {history[-1]['lejepa_loss']:.4f}")
print(f"History saved to {history_path}")

### 7.4 Learning curves

In [ ]:
ep = [r["epoch"] for r in history]
loss = [r["lejepa_loss"] for r in history]
inv = [r["inv_loss"] for r in history]
sig = [r["sigreg_loss"] for r in history]
lrs = [r["lr"] for r in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"LeJEPA - {ARCH} - ImageNet-100 - Lambda={LAMBDA}")

axes[0].plot(ep, loss, label="LeJEPA", color="black", lw=2)
axes[0].plot(ep, inv, label="Inv", ls="--")
axes[0].plot(ep, sig, label="SIGReg", ls=":")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="LeJEPA Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, [s / (l + 1e-8) for s, l in zip(sig, loss)], color="orange")
axes[1].axhline(LAMBDA, color="red", ls="--", label=f"lambda={LAMBDA}")
axes[1].set(xlabel="Epoch", title="SIGReg/Total ratio"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, lrs, color="green")
axes[2].axvline(WARMUP_EPOCHS, color="red", ls="--", label="End warmup")
axes[2].set(xlabel="Epoch", ylabel="LR", title="LR schedule")
axes[2].set_yscale("log"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(CKPT_DIR, "training_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved in {plot_path}")

## 8. Linear probe

In [ ]:
# load final backbone
final_bb_path = os.path.join(CKPT_DIR, f'backbone_ep{EPOCHS:03d}.pt')
state = torch.load(final_bb_path, map_location=DEVICE)
backbone.load_state_dict(state["backbone_state"])
print(f"Backbone ep{EPOCHS} loaded")

# freeze backbone
backbone.eval()
for p in backbone.parameters():
    p.requires_grad_(False)

# features for linear probe
# ViT: concat CLS token of the last two block
# ResNet: direct output after global average pooling
PROBE_HOOK_LAYERS = {
    "resnet50": None,
    "vit_small_patch16_224": ["blocks.10", "blocks.11"]
}
PROBE_DIM = EMBED_DIM * 2 if IS_VIT else EMBED_DIM


def extract_probe_features(backbone, x, arch):
    """Extract features for linear probe"""
    if not IS_VIT:
        return get_cls_embedding(backbone, x, arch) # (B, EMBED_DIM)

    feats, hooks = {}, []
    modules = dict(backbone.named_modules())
    for name in PROBE_HOOK_LAYERS[arch]:
        layer = modules[name]
        hooks.append(layer.register_forward_hook(
            lambda m, i, o, n=name: feats.__setitem__(n, o)
        ))
    try:
        _ = backbone(x)
    finally:
        for h in hooks:
            h.remove()

    cls_tokens = [
        feats[n][:, 0, :] # CLS
        for n in PROBE_HOOK_LAYERS[arch] 
    ] # 2 x (B, EMBED_DIM)

    return torch.cat(cls_tokens, dim=-1) #(B, 2*EMBED_DIM)

@torch.no_grad()
def extract_all_features(backbone, loader):
    """Extract features for linear probe for all split"""
    feats, labels = [], []
    amp_dtype = torch.bfloat16 if AMP_DTYPE == "bfloat16" else torch.float16
    for images, lbls in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=USE_AMP):
            f = extract_probe_features(backbone, images, ARCH)
        feats.append(f.float().cpu())
        labels.append(lbls)
    return torch.cat(feats), torch.cat(labels)


# train dataset without augmentation
train_probe_dataset = ImageFolder(IMAGENET_ROOT_TRAIN, transform=val_transform)
train_probe_loader = torch.utils.data.DataLoader(
    train_probe_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

train_feats, train_labels = extract_all_features(backbone, train_probe_loader)
val_feats, val_labels = extract_all_features(backbone, val_loader)
print(f'Train features: {tuple(train_feats.shape)}')
print(f'Val features: {tuple(val_feats.shape)}')

# probe = LayerNorm(PROBE_DIM) + Linear(PROBE_DIM, NUM_CLASSES)
# AdamW, weight decay 1e-6, linear warmup + cosine annealing
PROBE_EPOCHS = 20
PROBE_WARMUP_EPOCHS = max(2, PROBE_EPOCHS // 10)

probe_model = nn.Sequential(
    nn.LayerNorm(PROBE_DIM),
    nn.Linear(PROBE_DIM, NUM_CLASSES)
).to(DEVICE)

probe_optimizer = optim.AdamW(
    probe_model.parameters(), lr=1e-3, weight_decay=1e-6
)
probe_warmup = optim.lr_scheduler.LinearLR(
    probe_optimizer, start_factor=1e-3, end_factor=1.0, total_iters=PROBE_WARMUP_EPOCHS
)
probe_cosine = optim.lr_scheduler.CosineAnnealingLR(
    probe_optimizer, T_max=PROBE_EPOCHS-PROBE_WARMUP_EPOCHS, eta_min=1e-5
)
probe_scheduler = optim.lr_scheduler.SequentialLR(
    probe_optimizer, 
    schedulers=[probe_warmup, probe_cosine], 
    milestones=[PROBE_WARMUP_EPOCHS]
)
criterion = nn.CrossEntropyLoss()

train_ds = torch.utils.data.TensorDataset(train_feats, train_labels)
val_ds = torch.utils.data.TensorDataset(val_feats, val_labels)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=512, shuffle=True)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=512, shuffle=False)

print("Linear probe...")
best_acc = 0.0
# training
for ep in range(PROBE_EPOCHS):
    probe_model.train()
    for f, l in train_dl:
        f, l = f.to(DEVICE), l.to(DEVICE)
        probe_optimizer.zero_grad()
        criterion(probe_model(f), l).backward()
        probe_optimizer.step()
    probe_scheduler.step()

    # evaluation
    probe_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for f, l in val_dl:
            f, l  = f.to(DEVICE), l.to(DEVICE)
            correct += (probe_model(f).argmax(1) == l).sum().item()
            total += l.size(0)
    acc = 100.0 * correct / total
    if acc > best_acc:
        best_acc = acc
        # save best head
        torch.save({"head_state": probe_model.state_dict(),
                    "probe_dim": PROBE_DIM,
                    "val_acc1": best_acc},
                   os.path.join(CKPT_DIR, "linear_head.pt"))
    print(f'ep {ep+1:2d}/{PROBE_EPOCHS}: acc1 {acc:.2f}%')

print(f'Best linear probe acc1: {best_acc:.2f}%')

torch.save({
    "backbone_state": backbone.state_dict(),
    "arch": ARCH, "embed_dim": EMBED_DIM,
    "epoch": EPOCHS, "seed": SEED,
    "hook_layers": HOOK_LAYERS[ARCH],
    "val_acc1": best_acc,
}, os.path.join(CKPT_DIR, "best_backbone.pt"))

print(f"Saved in {CKPT_DIR}:")
print(f"best_backbone.pt (backbone frozen, with val_acc1)")
print(f"linear_head.pt (probe: LayerNorm + Linear)")